In [1]:
import sys
import os

project_root = os.path.abspath("..")
sys.path.append(project_root)

print(project_root)

c:\Users\SWS\Desktop\summer\projects\credit fraud


In [2]:
from src.preprocessing import (
    clean_data,
    PREPROCESSING_PIPELINES
)
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

In [3]:
df= pd.read_csv("../data/creditcard.csv")

In [4]:
# Clean data
df = clean_data(df)

# Split X / y
X = df.drop(columns=["Class"])
y = df["Class"]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [5]:
results = []

for pipeline_name, preprocessor in PREPROCESSING_PIPELINES.items():

    model = Pipeline([
        ("preprocessing", preprocessor),
        ("classifier", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        ))
    ])

    # Train
    model.fit(X_train, y_train)

    # Predictions
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    # Metrics
    results.append({
        "pipeline": pipeline_name,
        "AUPRC": average_precision_score(y_test, y_proba),
        "ROC_AUC": roc_auc_score(y_test, y_proba),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred)
    })

c:\Users\SWS\Desktop\summer\projects\credit fraud\credit\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [6]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="AUPRC",
    ascending=False
)

results_df

,pipeline,AUPRC,ROC_AUC,Precision,Recall,F1
3,linear_time_cyclic_scaled,0.680293,0.963500,0.055855,0.873684,0.104997
1,tree_time_cyclic,0.678536,0.963467,0.055930,0.873684,0.105130
2,linear_scaled,0.676968,0.962617,0.055370,0.873684,0.104141
0,raw,0.670647,0.962751,0.051681,0.873684,0.097590


#### Key Findings

- The **Time Cyclic** pipeline achieved the best overall performance with an **AUPRC of 0.680**, making it the most effective preprocessing strategy among those tested.
- The **Scaled** pipeline produced very similar results, indicating that feature scaling contributes positively to model performance.
- The **Raw** pipeline performed reasonably well, suggesting that the original PCA-transformed features already contain substantial predictive information.
- Surprisingly, the **Log Amount** pipeline yielded the lowest AUPRC, indicating that applying a logarithmic transformation to the transaction amount alone does not improve Logistic Regression performance on this dataset.

#### Recall Analysis

All pipelines achieved approximately **87.4% recall**, meaning that the model successfully detected nearly nine out of ten fraudulent transactions.

This is desirable in fraud detection because missing a fraudulent transaction is generally more costly than incorrectly flagging a legitimate one.

#### Precision Analysis

Precision remains very low (around **5%** for all pipelines). This means that among all transactions flagged as fraudulent, only about 5% are actually frauds.

This behavior is common in highly imbalanced fraud detection datasets and indicates that the model generates a large number of false positives.

#### ROC-AUC Analysis

All pipelines achieved a **ROC-AUC around 0.963**, indicating strong ranking ability and confirming that Logistic Regression can effectively separate fraudulent and legitimate transactions.

#### Conclusion

The introduction of cyclical time features (Hour_sin and Hour_cos) provides a measurable improvement over the baseline and produces the best overall performance. Consequently, the **Time Cyclic pipeline will be retained as the preferred preprocessing strategy for subsequent experiments with more advanced models such as Random Forest and XGBoost.**